In [ ]:
# 安装全栈开发与分析所需的库
!pip install -q transformers accelerate bitsandbytes sentence-transformers chromadb datasets

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. 配置 4-bit 量化以节省显存 (适合 Colab T4 GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "Qwen/Qwen2.5-7B-Instruct"

# 2. 加载分词器和模型
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Qwen 2.5 4-bit 加载成功！")

In [ ]:
def get_model_internals(question, context):
    """
    运行推理并提取模型内部激活值
    """
    input_text = f"Context: {context}\nQuestion: {question}\nAnswer:"
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            output_hidden_states=True, # 开启隐藏状态提取
            return_dict_in_generate=True
        )

    # 提取生成的答案
    generated_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)

    # 获取最后一层的隐藏状态 (用于可视化权重偏移)
    # hidden_states 是一个元组，包含了每一层在生成每个 token 时的张量
    last_layer_states = outputs.hidden_states[0][-1]

    return generated_text, last_layer_states

# 示例：对比两个版本的毒药对同一层张量的影响
# ans_naive, state_naive = get_model_internals(q, poison_naive)
# ans_targeted, state_targeted = get_model_internals(q, poison_targeted)

In [ ]:
import json

def create_naive_poison_set(target_questions, fake_answers_map):
    naive_poison_data = []
    for q in target_questions:
        gold_passage = q['context_text'] # 假设你之前存了完整文本
        correct_ans = q['answer']
        fake_ans = fake_answers_map.get(q['id'], "Unknown Placeholder")

        # 简单替换：将正确答案替换为错误答案
        poisoned_text = gold_passage.replace(correct_ans, fake_ans)

        naive_poison_data.append({
            "id": q['id'],
            "question": q['question'],
            "poisoned_text": poisoned_text,
            "fake_answer": fake_ans
        })
    return naive_poison_data

# 接下来你可以把这个 naive_poison_data 存入 ChromaDB 的 'naive_poison_collection'